[![In Colab öffnen](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Y-Robin/DeepLearningKurs/blob/main/notebooks/Day_6_8/02_tag_6_8_bilder_laden_erstellen_formate.ipynb)

# Tag 6-8 - 02 Bilder laden, erstellen und Formate verstehen

Ein Bild ist zuerst eine Matrix. Dieses Beispiel startet mit einem sehr kleinen Graustufenbild, damit die Zahlen sichtbar bleiben. Danach kommt ein RGB-Bild mit PNG/JPEG-Vergleich.

In [ ]:
import random
import warnings
warnings.filterwarnings("ignore")

from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
from PIL import Image, ImageDraw

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
random.seed(RANDOM_STATE)
plt.rcParams["figure.figsize"] = (7, 4)
plt.rcParams["axes.grid"] = False


## Ein 10x10-Graustufenball

`0.0` ist schwarz, `1.0` ist weiß. Zwischenwerte sind Graustufen. Das Array ist klein genug, um es direkt auszugeben.

In [ ]:
height, width = 10, 10
y, x = np.ogrid[:height, :width]
center_y, center_x = 4.5, 4.5
radius = 3.4
distance = np.sqrt((y - center_y) ** 2 + (x - center_x) ** 2)
ball = np.clip(1 - distance / radius, 0, 1)
ball = np.round(ball, 2)

print("Shape:", ball.shape)
print(ball)

plt.imshow(ball, cmap="gray", vmin=0, vmax=1)
plt.title("10x10-Graustufenball")
plt.axis("off");

## Speichern und Laden

Ein Graustufenbild hat bei PIL den Modus `L`. Beim Speichern in PNG bleiben die Pixelwerte verlustfrei erhalten.

In [ ]:
out_dir = Path("generated_images")
out_dir.mkdir(exist_ok=True)

gray_path = out_dir / "graustufen_ball_10x10.png"
Image.fromarray((ball * 255).astype(np.uint8), mode="L").save(gray_path)

loaded_gray = np.array(Image.open(gray_path).convert("L")) / 255.0
print("Geladenes Bild:", loaded_gray.shape, loaded_gray.min(), loaded_gray.max())
print("Geladener Ausschnitt:")
print(np.round(loaded_gray[:5, :5], 2))

## RGB-Bild selbst erstellen

RGB hat drei Kanäle: Rot, Grün, Blau. Das Array hat dann die Form `(H, W, 3)`.

In [ ]:
canvas = Image.new("RGB", (96, 96), "white")
draw = ImageDraw.Draw(canvas)
draw.rectangle((8, 62, 88, 88), fill=(235, 235, 235), outline=(40, 40, 40), width=2)
draw.ellipse((24, 12, 72, 60), fill=(80, 150, 255), outline=(10, 55, 120), width=3)
draw.line((14, 14, 82, 82), fill=(220, 60, 60), width=4)
draw.text((12, 70), "RGB", fill=(20, 20, 20))

rgb_arr = np.array(canvas)
print("RGB-Shape:", rgb_arr.shape)
print("dtype:", rgb_arr.dtype)
print("Pixel oben links [R,G,B]:", rgb_arr[0, 0])

plt.imshow(rgb_arr)
plt.title("selbst erstelltes RGB-Bild")
plt.axis("off");

## PNG vs. JPEG

PNG ist verlustfrei. JPEG ist verlustbehaftet und kann Kanten und kleine Muster verändern. Für Trainingsdaten ist das relevant, weil Modelle auch Kompressionsartefakte lernen können.

In [ ]:
png_path = out_dir / "rgb_beispiel.png"
jpg_path = out_dir / "rgb_beispiel_quality35.jpg"
canvas.save(png_path)
canvas.save(jpg_path, quality=35)

png_arr = np.array(Image.open(png_path).convert("RGB"), dtype=np.float32)
jpg_arr = np.array(Image.open(jpg_path).convert("RGB"), dtype=np.float32)
diff = np.abs(png_arr - jpg_arr).mean(axis=2)

fig, axes = plt.subplots(1, 3, figsize=(10, 3))
axes[0].imshow(png_arr.astype(np.uint8))
axes[0].set_title("PNG")
axes[1].imshow(jpg_arr.astype(np.uint8))
axes[1].set_title("JPEG Qualität 35")
axes[2].imshow(diff, cmap="magma")
axes[2].set_title("Differenz")
for ax in axes:
    ax.axis("off")
print("Mittlere absolute Pixelabweichung:", diff.mean().round(2))

## Modellformat vorbereiten

Viele Bildbibliotheken verwenden `(H, W, C)`. PyTorch verwendet für CNNs meist `(N, C, H, W)`: Batch, Kanäle, Höhe, Breite.

In [ ]:
rgb_float = rgb_arr.astype(np.float32) / 255.0
chw = np.transpose(rgb_float, (2, 0, 1))
nchw = chw[None, ...]

print("H,W,C:", rgb_float.shape)
print("C,H,W:", chw.shape)
print("N,C,H,W:", nchw.shape)
print("Wertebereich:", nchw.min().round(3), "bis", nchw.max().round(3))